In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# split para modelado
from sklearn.model_selection import train_test_split
# Scaled | Escalado
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Encoding | Codificación
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
# To save models
import json
import pickle
# Feature Selection
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.metrics.pairwise import cosine_similarity

RecursionError: maximum recursion depth exceeded

In [ ]:
df = pd.read_csv("../data/raw/census-income.csv")
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [ ]:
# obtener las dimenciones
df.shape

(32561, 15)

In [ ]:
# Obtener información sobre tipos de datos y valores no nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,32561.0,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
fnlwgt,32561.0,189778.366512,105549.977697,12285.0,117827.0,178356.0,237051.0,1484705.0
education.num,32561.0,10.080679,2.572720,1.0,9.0,10.0,12.0,16.0
capital.gain,32561.0,1077.648844,7385.292085,0.0,0.0,0.0,0.0,99999.0
capital.loss,32561.0,87.303830,402.960219,0.0,0.0,0.0,0.0,4356.0
hours.per.week,32561.0,40.437456,12.347429,1.0,40.0,40.0,45.0,99.0


In [ ]:
df.duplicated().sum()

np.int64(24)

In [ ]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'income'],
      dtype='object')

### Limpieza de datos

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [ ]:
df = df.drop(columns=["fnlwgt", "capital.gain", "capital.loss", "marital.status", "relationship", "education"])
df.head(5)

,age,workclass,education.num,occupation,race,sex,hours.per.week,native.country,income
0,90,?,9,?,White,Female,40,United-States,<=50K
1,82,Private,9,Exec-managerial,White,Female,18,United-States,<=50K
2,66,?,10,?,Black,Female,40,United-States,<=50K
3,54,Private,4,Machine-op-inspct,White,Female,40,United-States,<=50K
4,41,Private,10,Prof-specialty,White,Female,40,United-States,<=50K


##### Cambiar valores

In [ ]:
df.replace('?', np.nan, inplace=True)

In [ ]:
df.dropna(inplace=True)
df.shape

(30139, 9)

In [ ]:
df['income']

1        <=50K
3        <=50K
4        <=50K
5        <=50K
6        <=50K
         ...  
32556    <=50K
32557    <=50K
32558     >50K
32559    <=50K
32560    <=50K
Name: income, Length: 30139, dtype: object

In [ ]:
df['income'] = df['income'].str.strip()
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

In [ ]:
df['income']

1        0
3        0
4        0
5        0
6        0
        ..
32556    0
32557    0
32558    1
32559    0
32560    0
Name: income, Length: 30139, dtype: int64

In [ ]:
# realizando one-hot-encoding para pasar a numerico las variables de clasificacion
categorical_cols = ["workclass","occupation", "sex", "race", "native.country"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, dtype=int)
df_encoded

,age,education.num,hours.per.week,income,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
1,82,9,18,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,54,4,40,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,41,10,40,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
5,34,9,45,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
6,38,6,40,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,10,40,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
32557,27,12,38,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
32558,40,9,40,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
32559,58,9,40,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
# Normalizacion de las variables numericas

scaler = StandardScaler()

df_encoded[['age', 'hours.per.week','education.num']] = scaler.fit_transform(
    df_encoded[['age', 'hours.per.week', 'education.num']])
df_encoded.head()

,age,education.num,hours.per.week,income,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
1,3.317157,-0.440434,-1.914647,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1.184832,-2.402221,-0.078031,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,0.194824,-0.048076,-0.078031,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
5,-0.338257,-0.440434,0.339381,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
6,-0.033639,-1.617506,-0.078031,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0


#### Split

In [ ]:
X = df_encoded.drop(columns=['income'])
y = df_encoded['income']

: 

In [ ]:
# calcular similitudes entre usuarios
#similarity_matrix = cosine_similarity(X)


NameError: name 'cosine_similarity' is not defined

In [ ]:


# Funcion de recomendacion
# Busca personas similares que ganan mas de 50k y recomienda sus caracteristicas

def recomendar_trayectoria(usuario_idx, df_original, X, y, top_k=5):
    sims = similarity_matrix[usuario_idx]
    
    similar_users = sims.argsort()[::-1][1:]
    similares_altos_ingresos = [
        i for i in similar_users if y.iloc[i] == 1
    ][:top_k]
    
    return df_original.iloc[similares_altos_ingresos][
        ['education.num', 'occupation', 'hours.per.week']
    ]



In [ ]:
# Prueba simple: usar un usuario real del dataset
# Paso 1: elegir un usuario con ingreso bajo del mismo dataset

usuario_test = df[df['income'] == 0].index[0]


In [ ]:

df.loc[usuario_test]

In [ ]:
#recomendar_trayectoria(usuario_test, df, X, y)